# Fraud Detection: Hyperparameter Tuning (LightGBM)

Milestone 3. In [08_model-comparison.ipynb](08_model-comparison.ipynb), **LightGBM** had the highest ROC-AUC of all default models (CV 0.766 ± 0.004, validation 0.767), so it is the model we tune (decision in [protocol.md](protocol.md)).

**Goal:** find LightGBM settings that rank fraud cases better than the default settings.

**How:**
- **Random search** over the settings, scored with **ROC-AUC** (our main metric), PR-AUC is reported next to it.
- **5-fold stratified cross-validation on the training split only**, with the same folds as in notebook 08, so the scores are directly comparable.
- The fraud class stays weighted (`class_weight="balanced"`, team decision).
- The **validation split is used only once**, at the end, to check the tuned model against the default.

The notebook shows only tables and numbers, no charts.

In [1]:
# libraries for data handling, the model, the random search and the metrics
import json
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate

pd.set_option("display.max_columns", 40)
FEATURE_DIR = Path("data/features")
MODEL_DIR = Path("models")
RANDOM_STATE = 42

## 1. Load the feature tables

LightGBM uses the **tree tables** from [06_feature-engineering.ipynb](06_feature-engineering.ipynb): 24 columns, unscaled, categories as integers, NaN kept. Index = `client_id`.

In [2]:
# read one feature table from data/features, with client_id as the index
def load(name):
    return pd.read_csv(FEATURE_DIR / f"{name}.csv", index_col="client_id")

X_train, X_valid = load("X_train"), load("X_valid")
y_train, y_valid = load("y_train")["target"], load("y_valid")["target"]

# features and labels must belong to the same clients, in the same order
assert (X_train.index == y_train.index).all() and (X_valid.index == y_valid.index).all()

print("train:", X_train.shape, f"fraud {y_train.mean():.2%}")
print("valid:", X_valid.shape, f"fraud {y_valid.mean():.2%}")

train: (108394, 24) fraud 5.58%
valid: (27099, 24) fraud 5.58%


## 2. Scoring helper

The same helper as in notebooks 07 and 08. It takes the fraud probabilities of the validation clients and reports ROC-AUC, PR-AUC and precision, recall and F1 at the F1-best threshold (chosen on the validation data, so an upper bound).

In [3]:
def score(y_true, proba):
    # precision and recall for every possible threshold
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # F1 for every threshold; the small number avoids dividing by zero
    f1_curve = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    # the F1-best threshold: the cut-off with the highest F1 on this data
    best = f1_curve.argmax()
    return {
        "valid ROC-AUC": roc_auc_score(y_true, proba),
        "valid PR-AUC": average_precision_score(y_true, proba),
        "best threshold": thresholds[best],
        "precision@best": precision[best],
        "recall@best": recall[best],
        "F1@best": f1_curve[best],
    }

## 3. Reference: LightGBM with default settings

The score to beat. Same model and same 5 folds as in notebook 08, so the CV ROC-AUC should be 0.766 again.

`train ROC-AUC` is the score on the clients the model was fitted on; the **gap** to the CV score is our overfitting check.

In [4]:
# the same 5 stratified folds as in notebook 08
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# default LightGBM, fraud weighted as decided
default_model = LGBMClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
default_cv = cross_validate(default_model, X_train, y_train, cv=cv,
                            scoring=["roc_auc", "average_precision"], return_train_score=True)

print(f"default CV ROC-AUC: {default_cv['test_roc_auc'].mean():.4f} (± {default_cv['test_roc_auc'].std():.4f})")
print(f"default CV PR-AUC:  {default_cv['test_average_precision'].mean():.4f}")
print(f"default train ROC-AUC: {default_cv['train_roc_auc'].mean():.4f}")

default CV ROC-AUC: 0.7656 (± 0.0037)
default CV PR-AUC:  0.1695
default train ROC-AUC: 0.8608


## 4. Random search

A random search tries **30 random combinations** of settings instead of every combination, which is much faster with this many settings. We do not use a grid search: even 3 values for each of the 8 settings give 3^8 = 6,561 combinations, about 33,000 model fits or roughly 1 to 2 days, and a random search covers the important settings with far fewer fits. Each combination is scored with the same 5 folds, so this means 150 model fits. The best combination (highest mean CV ROC-AUC) is then refitted on the whole training split.

| Setting | What it does | Range |
|---|---|---|
| `n_estimators` | number of trees in the line | 100 to 1000 |
| `learning_rate` | how much each new tree may correct; smaller = slower but more careful | 0.01 to 0.15 |
| `num_leaves` | how many end points (leaves) one tree may have; more = more complex | 15 to 127 |
| `max_depth` | how deep one tree may grow (-1 = no limit) | -1, 4, 6, 8 |
| `min_child_samples` | a leaf needs at least this many clients; higher = calmer model | 20 to 300 |
| `subsample` | each tree sees only this share of the clients, which reduces overfitting | 0.6 to 1.0 |
| `colsample_bytree` | each tree sees only this share of the columns | 0.5 to 1.0 |
| `reg_lambda` | penalty on extreme leaf values; higher = calmer model | 0.001 to 10 |

The class weight is **not** searched: it stays `"balanced"` (team decision). `subsample_freq=1` switches the client sampling on for every tree.

In [5]:
# the range for each setting; the search draws 30 random combinations from it
param_distributions = {
    "n_estimators": randint(100, 1001),
    "learning_rate": uniform(0.01, 0.14),        # 0.01 to 0.15
    "num_leaves": randint(15, 128),
    "max_depth": [-1, 4, 6, 8],
    "min_child_samples": randint(20, 301),
    "subsample": uniform(0.6, 0.4),              # 0.6 to 1.0
    "colsample_bytree": uniform(0.5, 0.5),       # 0.5 to 1.0
    "reg_lambda": loguniform(1e-3, 10),
}

search = RandomizedSearchCV(
    LGBMClassifier(class_weight="balanced", subsample_freq=1, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    param_distributions,
    n_iter=30,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
    refit="roc_auc",                # choose and refit by ROC-AUC, our main metric
    cv=cv,
    return_train_score=True,        # needed for the overfitting check
    random_state=RANDOM_STATE,
    verbose=1,
)
# fit on the training split only; the validation split is not touched here
search.fit(X_train, y_train)

print(f"best CV ROC-AUC: {search.best_score_:.4f}")
# the best settings, rounded for reading
{k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in search.best_params_.items()}

Fitting 5 folds for each of 30 candidates, totalling 150 fits


best CV ROC-AUC: 0.7687


{'colsample_bytree': 0.8676,
 'learning_rate': 0.0393,
 'max_depth': 6,
 'min_child_samples': 118,
 'n_estimators': 362,
 'num_leaves': 30,
 'reg_lambda': 1.8573,
 'subsample': 0.8221}

In [6]:
# the 5 best combinations: is the top stable, or one lucky draw?
results = pd.DataFrame(search.cv_results_)
results["gap"] = results["mean_train_roc_auc"] - results["mean_test_roc_auc"]
columns = ["mean_test_roc_auc", "std_test_roc_auc", "mean_test_pr_auc", "gap"] + [c for c in results.columns if c.startswith("param_")]
results.sort_values("mean_test_roc_auc", ascending=False)[columns].head(5).round(4)

,mean_test_roc_auc,std_test_roc_auc,mean_test_pr_auc,gap,param_colsample_bytree,param_learning_rate,param_max_depth,param_min_child_samples,param_n_estimators,param_num_leaves,param_reg_lambda,param_subsample
28,0.7687,0.0042,0.1751,0.0926,0.8676,0.0393,6,118,362,30,1.8573,0.8221
23,0.7683,0.0034,0.1724,0.0744,0.9883,0.0675,6,243,324,15,0.0091,0.6580
26,0.7673,0.0038,0.1723,0.0928,0.9177,0.0549,4,278,759,73,0.0076,0.8845
2,0.7670,0.0043,0.1675,0.0517,0.9693,0.0101,8,296,260,90,0.2796,0.6028
4,0.7670,0.0045,0.1740,0.1126,0.8926,0.0380,6,263,931,17,2.7527,0.8721


## 5. Default vs tuned (cross-validation)

Both on the same 5 folds of the training split.

In [7]:
# the CV scores of the best combination, taken from the search results
best = results.loc[search.best_index_]

cv_comparison = pd.DataFrame({
    "LightGBM default": {
        "CV ROC-AUC": default_cv["test_roc_auc"].mean(),
        "CV ROC-AUC std": default_cv["test_roc_auc"].std(),
        "CV PR-AUC": default_cv["test_average_precision"].mean(),
        "train ROC-AUC": default_cv["train_roc_auc"].mean(),
    },
    "LightGBM tuned": {
        "CV ROC-AUC": best["mean_test_roc_auc"],
        "CV ROC-AUC std": best["std_test_roc_auc"],
        "CV PR-AUC": best["mean_test_pr_auc"],
        "train ROC-AUC": best["mean_train_roc_auc"],
    },
}).T
cv_comparison["gap"] = cv_comparison["train ROC-AUC"] - cv_comparison["CV ROC-AUC"]

gain = cv_comparison.loc["LightGBM tuned", "CV ROC-AUC"] - cv_comparison.loc["LightGBM default", "CV ROC-AUC"]
print(f"improvement in CV ROC-AUC: {gain:+.4f}")
cv_comparison.round(4)

improvement in CV ROC-AUC: +0.0031


,CV ROC-AUC,CV ROC-AUC std,CV PR-AUC,train ROC-AUC,gap
LightGBM default,0.7656,0.0037,0.1695,0.8608,0.0951
LightGBM tuned,0.7687,0.0042,0.1751,0.8613,0.0926


## 6. Final check on the validation split

The validation clients have not been used for anything so far. We use them **once**: the default model is fitted on the whole training split, the tuned model is the refitted best model of the search. If the tuned model is also better here, the improvement is real and not a result of choosing among 30 combinations.

In [8]:
# default model fitted on all training clients
default_model.fit(X_train, y_train)
tuned_model = search.best_estimator_

valid_comparison = pd.DataFrame({
    "LightGBM default": score(y_valid, default_model.predict_proba(X_valid)[:, 1]),
    "LightGBM tuned": score(y_valid, tuned_model.predict_proba(X_valid)[:, 1]),
}).T
valid_comparison.round(4)

,valid ROC-AUC,valid PR-AUC,best threshold,precision@best,recall@best,F1@best
LightGBM default,0.7669,0.1780,0.6345,0.1641,0.4448,0.2397
LightGBM tuned,0.7742,0.1863,0.7157,0.2170,0.3034,0.2530


## 7. Save the tuned model

The tuned model and its settings are saved in `models/`, so the next notebooks (error analysis, final prediction for the Zindi test set) can use them without searching again.

In [9]:
MODEL_DIR.mkdir(exist_ok=True)

# the trained trees in LightGBM's own text format
tuned_model.booster_.save_model(str(MODEL_DIR / "lightgbm_tuned.txt"))

# the chosen settings, including the fixed ones, so the model can be rebuilt exactly
params = {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in search.best_params_.items()}
params.update({"class_weight": "balanced", "subsample_freq": 1, "random_state": RANDOM_STATE})
(MODEL_DIR / "lightgbm_best_params.json").write_text(json.dumps(params, indent=2))

print(sorted(p.name for p in MODEL_DIR.glob("lightgbm_*")))

['lightgbm_best_params.json', 'lightgbm_tuned.txt']


## Result

| LightGBM (weighted) | CV ROC-AUC (± std) | CV PR-AUC | Gap | Valid ROC-AUC | Valid PR-AUC |
|---|---|---|---|---|---|
| Default settings | 0.766 (± 0.004) | 0.170 | 0.095 | 0.767 | 0.178 |
| **Tuned** | **0.769** (± 0.004) | **0.175** | 0.093 | **0.774** | **0.186** |

- **Tuning helps a little.** In cross-validation the tuned model is +0.003 ROC-AUC better. That is smaller than the spread between the folds (about 0.004), so on its own it is not a clear improvement.
- **The validation split confirms the direction**: +0.007 ROC-AUC and +0.008 PR-AUC, on clients that were not used for the search. Tuned is better on both checks and never worse, so we use the **tuned model** from here on.
- **The top of the search is flat**: the 5 best combinations all score between 0.767 and 0.769. Different settings give almost the same result, so more searching (or a grid search) would not bring much. The default LightGBM was already close to its best.
- **No extra overfitting**: the gap stays at about 0.09. The tuned model uses fewer and smaller trees than it could (362 trees, 30 leaves, depth 6, at least 118 clients per leaf), which keeps it calm.
- **Business view** (F1-best threshold on validation, upper bound): the tuned model finds 30% of the fraud cases with 22% precision (F1 0.25). The default model at its own best threshold found more fraud (44%) but with lower precision (16%). The real cut-off still has to follow STEG's inspection capacity.

**Saved:** `models/lightgbm_tuned.txt` (the trained model) and `models/lightgbm_best_params.json` (the settings).

**Next steps:** error analysis of the tuned model (which fraud cases does it miss, and what do they have in common?), then refit with these settings on train plus validation and predict the Zindi test set.